# 04. Speech Emotion Recognition


## What This Notebook Does

This notebook uses a very basic speech ML pipeline:

1. collect audio files
2. convert each file into a numeric feature vector
3. split by speaker so the test set stays honest
4. scale the features
5. train one SVM model
6. save the trained files


## Step 1: Setup


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()
possible_dirs = [current_dir, current_dir / "NeuroSense" / "notebooks"]
notebooks_dir = next((path for path in possible_dirs if path.exists() and path.name == "notebooks"), None)
if notebooks_dir is None:
    raise FileNotFoundError("Start Jupyter from the project root or from NeuroSense/notebooks.")

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook()
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print("Datasets directory:", DATASETS_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)


## Step 2: Import The Libraries


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from notebook_support import collect_audio_paths, dedupe_records_by_content, extract_feature_dataset, label_counts, sample_records_by_label
from utils.emotion_utils import map_emotion_to_sentiment
from utils.preprocessors import preprocess_speech


## Step 3: Collect Audio Files

We read the audio paths, convert folder names into labels, remove exact duplicate files, and keep a small balanced subset so the notebook stays fast.


In [ ]:
def normalize_speech_label(folder_name):
    folder_name = folder_name.lower().replace("-", "_").replace(" ", "_")
    if "pleasant" in folder_name or folder_name.endswith("_ps") or folder_name == "ps":
        return "ps"
    for emotion in ("angry", "disgust", "fear", "happy", "neutral", "sad"):
        if emotion in folder_name:
            return emotion
    return None


speech_root = DATASETS_DIR / "speech"
if not speech_root.exists():
    raise FileNotFoundError(f"Missing speech dataset directory: {speech_root}")

raw_records = collect_audio_paths(speech_root, normalize_speech_label)
records = dedupe_records_by_content(raw_records)
records = sample_records_by_label(records, per_label=150, seed=RANDOM_STATE)

print("Audio files found:", len(records))
print("Emotion label counts:", label_counts(label for _, label in records))
print("Sentiment counts:", label_counts(map_emotion_to_sentiment(label) for _, label in records))


## Step 4: Extract Features

Each audio file is converted into one fixed-size numeric vector.


In [ ]:
X, y_raw = extract_feature_dataset(
    records,
    preprocess_speech,
    cache_path=CACHE_DIR / "speech_features_deduped.npz",
    progress_interval=100,
)

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

speaker_groups = np.array([path.parent.name.split("_")[0].upper() for path, _ in records])
unique_speakers = sorted(set(speaker_groups))
if len(unique_speakers) < 2:
    raise ValueError("This notebook needs at least two speakers for a speaker-holdout split.")

train_speaker = unique_speakers[0]
test_speaker = unique_speakers[1]

train_mask = speaker_groups == train_speaker
test_mask = speaker_groups == test_speaker

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print("Feature matrix shape:", X.shape)
print("Class names:", list(encoder.classes_))
print("Train speaker:", train_speaker, "- samples:", X_train.shape[0])
print("Test speaker:", test_speaker, "- samples:", X_test.shape[0])


## Step 5: Scale The Features

We fit the scaler only on the training speaker, then reuse it for the test speaker.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 6: Train The Model


In [ ]:
model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)


## Step 7: Check The Result

We report the honest speaker-holdout result. We also print one random-split reference number so you can explain speaker leakage in viva.


In [ ]:
speaker_holdout_accuracy = accuracy_score(y_test, y_pred)
print("Speaker-holdout accuracy:", round(speaker_holdout_accuracy, 4))
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=encoder.classes_,
    cmap="magma",
    xticks_rotation=20,
)
plt.title("Speech confusion matrix (speaker holdout)")
plt.tight_layout()
plt.show()

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

random_scaler = StandardScaler()
X_train_random_scaled = random_scaler.fit_transform(X_train_random)
X_test_random_scaled = random_scaler.transform(X_test_random)

random_model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
random_model.fit(X_train_random_scaled, y_train_random)
random_accuracy = accuracy_score(y_test_random, random_model.predict(X_test_random_scaled))

print("Random split accuracy (reference only):", round(random_accuracy, 4))
print("This number can look too good because the same speaker may appear in both train and test.")


## Step 8: Save The Trained Files


In [ ]:
artifact_dir = ARTIFACTS_DIR / "speech"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "speech_model.pkl")
joblib.dump(scaler, artifact_dir / "speech_scaler.pkl")
joblib.dump(encoder, artifact_dir / "speech_label_encoder.pkl")

metadata = {
    "data_source": "Toronto Emotional Speech Set (TESS)",
    "data_source_note": "Audio clips are converted into fixed numeric features with librosa-based preprocessing.",
    "evaluation_method": f"Speaker Holdout ({train_speaker} train, {test_speaker} test)",
    "evaluation_note": "Speaker-holdout is the honest result because the test speaker is unseen during training.",
    "selected_model": "SVM (RBF)",
    "speaker_holdout_accuracy": round(float(speaker_holdout_accuracy), 4),
    "random_split_accuracy": round(float(random_accuracy), 4),
    "n_train": int(X_train.shape[0]),
    "n_test": int(X_test.shape[0]),
}
with open(artifact_dir / "speech_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved speech files to:", artifact_dir)
